# Análise Global de Mudanças Climáticas e Eventos Extremos
### Sistemas Distribuídos — UEL 2026
---
Este notebook realiza o processamento e análise de dados históricos de temperatura global
utilizando **Apache Spark** em cluster distribuído via Docker.

**Dataset principal:** GlobalLandTemperaturesByCity.csv (Berkeley Earth / Kaggle)  
**Dataset secundário:** owid-co2-data.csv (Our World in Data)

**Perguntas respondidas:**
1. Média Móvel de Temperatura por Década
2. Anomalias Locais — 10 Anos Mais Quentes por Continente
3. Cidades em Risco — Maior Desvio Padrão de Temperatura
4. Correlação de Estações em Zonas Tropicais
5. Qualidade de Dados — Filtro de Incerteza
6. Correlação entre Emissões de CO₂ e Aquecimento (Join)
7. Ranking de Aceleração Térmica (Window Functions)
8. Previsão de Tendência com Regressão Linear (MLlib)


## 1. Imports

In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col,
    year,
    month,
    to_date,
    avg,
    stddev,
    max as _max,
    min as _min,
    regexp_replace,
    expr,
    count,
    when,
    lag,
    round as _round,
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

## 2. Inicialização da Sessão Spark

Criamos uma `SparkSession` apontando para o master do cluster Docker.
O `setLogLevel("WARN")` reduz o volume de logs, exibindo apenas avisos e erros.


In [2]:
spark = (
    SparkSession.builder
    .appName("AnaliseClimaticaGlobal")
    .master("spark://spark-master:7077")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "/opt/spark/spark-events")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Sessão Spark iniciada com sucesso!")
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/26 16:34:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Sessão Spark iniciada com sucesso!


## 3. ETL — Ingestão e Limpeza de Dados

### 3.1 Leitura da base de temperaturas

Utilizamos o arquivo `GlobalLandTemperaturesByCity.csv` da Berkeley Earth.
O Spark infere o schema automaticamente com `inferSchema=True`.


In [3]:
df = spark.read.csv(
    "/opt/spark-data/input/GlobalLandTemperaturesByCity.csv",
    header=True,
    inferSchema=True,
)

print(f"Total de registros brutos: {df.count():,}")
df.printSchema()
df.show(5)

Total de registros brutos: 8,599,212
root
 |-- dt: date (nullable = true)
 |-- AverageTemperature: double (nullable = true)
 |-- AverageTemperatureUncertainty: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)

+----------+------------------+-----------------------------+-----+-------+--------+---------+
|        dt|AverageTemperature|AverageTemperatureUncertainty| City|Country|Latitude|Longitude|
+----------+------------------+-----------------------------+-----+-------+--------+---------+
|1743-11-01|             6.068|           1.7369999999999999|Århus|Denmark|  57.05N|   10.33E|
|1743-12-01|              NULL|                         NULL|Århus|Denmark|  57.05N|   10.33E|
|1744-01-01|              NULL|                         NULL|Århus|Denmark|  57.05N|   10.33E|
|1744-02-01|              NULL|                         NULL|Århus|Denmark|  57.05N|   1

### 3.2 Limpeza e transformações

Realizamos as seguintes operações:
- Conversão da coluna `dt` de String para `DateType`
- Extração das colunas `Year` e `Month` para facilitar as agregações
- Remoção de linhas com `AverageTemperature` nula
- Limpeza das coordenadas — remoção de letras N/S/E/W e conversão para `double`


In [4]:
df_limpo = (
    df.withColumn("dt", to_date(col("dt")))
    .withColumn("Year", year(col("dt")))
    .withColumn("Month", month(col("dt")))
    .dropna(subset=["AverageTemperature"])
    .withColumn("Latitude",  regexp_replace(col("Latitude"),  "[NSEW]", "").cast("double"))
    .withColumn("Longitude", regexp_replace(col("Longitude"), "[NSEW]", "").cast("double"))
)

print(f"Registros após remoção de nulos: {df_limpo.count():,}")
df_limpo.show(5)

Registros após remoção de nulos: 8,235,082
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+-----+
|        dt|AverageTemperature|AverageTemperatureUncertainty| City|Country|Latitude|Longitude|Year|Month|
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+-----+
|1743-11-01|             6.068|           1.7369999999999999|Århus|Denmark|   57.05|    10.33|1743|   11|
|1744-04-01|5.7879999999999985|           3.6239999999999997|Århus|Denmark|   57.05|    10.33|1744|    4|
|1744-05-01|            10.644|           1.2830000000000001|Århus|Denmark|   57.05|    10.33|1744|    5|
|1744-06-01|14.050999999999998|                        1.347|Århus|Denmark|   57.05|    10.33|1744|    6|
|1744-07-01|            16.082|                        1.396|Århus|Denmark|   57.05|    10.33|1744|    7|
+----------+------------------+-----------------------------+-----+-------+--------+---------+----+-----+
onl

### 3.3 Cache do DataFrame limpo

Fazemos `.cache()` no DataFrame limpo pois ele será reutilizado em todas as perguntas.
Sem cache, o Spark releria e reprocessaria o CSV do zero a cada nova ação.


In [5]:
df_limpo.cache()
print("Cache aplicado em df_limpo.")

Cache aplicado em df_limpo.


### 3.4 Leitura e limpeza da base de CO₂

Utilizamos o dataset `owid-co2-data.csv` (Our World in Data), que cobre desde 1750
e contém colunas como `co2_per_capita` e `total_ghg`, conforme especificado no enunciado.

Filtramos:
- Registros anteriores a 1900
- Entradas que representam agregados regionais/globais (ex: "World", "Asia (GCP)") que
  distorceriam as correlações por país
- Linhas com `co2` nulo


In [6]:
df_co2 = spark.read.csv(
    "/opt/spark-data/input/owid-co2-data.csv",
    header=True,
    inferSchema=True,
)

agregados_globais = [
    "World", "Asia", "Asia (GCP)", "Asia (excl. China and India)",
    "Europe", "Europe (GCP)", "Europe (excl. EU-27)", "Europe (excl. EU-28)",
    "European Union (27)", "European Union (28)",
    "Africa", "Africa (GCP)",
    "North America", "North America (GCP)", "North America (excl. USA)",
    "South America", "South America (GCP)", "Central America (GCP)",
    "Oceania", "Oceania (GCP)",
    "High-income countries", "Low-income countries",
    "Lower-middle-income countries", "Upper-middle-income countries",
    "OECD (GCP)", "OECD (Jones et al.)", "Non-OECD (GCP)",
]

df_co2_clean = (
    df_co2.filter(col("year") >= 1900)
    .filter(~col("country").isin(agregados_globais))
    .select(
        col("country").alias("Country_CO2"),
        col("year").alias("Year_CO2"),
        col("co2"),
        col("co2_per_capita"),
        col("total_ghg"),
    )
    .dropna(subset=["co2"])
)

print(f"Registros CO₂ após limpeza: {df_co2_clean.count():,}")
df_co2_clean.show(5)

Registros CO₂ após limpeza: 20,164
+-----------+--------+-----+--------------+---------+
|Country_CO2|Year_CO2|  co2|co2_per_capita|total_ghg|
+-----------+--------+-----+--------------+---------+
|Afghanistan|    1949|0.015|         0.002|   18.667|
|Afghanistan|    1950|0.084|         0.011|   19.869|
|Afghanistan|    1951|0.092|         0.012|   21.069|
|Afghanistan|    1952|0.092|         0.011|   22.094|
|Afghanistan|    1953|0.106|         0.013|   23.256|
+-----------+--------+-----+--------------+---------+
only showing top 5 rows



## 4. P5 — Qualidade de Dados: Filtro de Incerteza

Identificamos registros onde `AverageTemperatureUncertainty` é superior a **10% da média
histórica** de temperatura da cidade.

Usamos uma **Window Function** particionada por `City` e `Country` para calcular a média
histórica de cada cidade — sem precisar de um join separado.

Após identificar os registros problemáticos, criamos `df_filtrado` que será usado
em todas as perguntas seguintes.


In [7]:
# Especificação da janela por cidade
window_city = Window.partitionBy("City", "Country")

# Adicionando coluna com a média histórica de cada cidade
df_limpo_com_histavg = df_limpo.withColumn(
    "HistAvg", avg("AverageTemperature").over(window_city)
)

# Registros com alta incerteza (serão removidos)
df_alta_incerteza = df_limpo_com_histavg.filter(
    col("AverageTemperatureUncertainty") > (0.10 * col("HistAvg"))
)

print("P5 — Top 10 países com mais registros de alta incerteza:")
df_alta_incerteza.groupBy("Country").count().orderBy(col("count").desc()).show(10)

# DataFrame filtrado para uso nas demais perguntas
df_filtrado = df_limpo_com_histavg.filter(
    col("AverageTemperatureUncertainty") <= (0.10 * col("HistAvg"))
)

total = df_filtrado.count()
print(f"Registros restantes após filtro de incerteza: {total:,}")

P5 — Top 10 países com mais registros de alta incerteza:


+--------------+------+
|       Country| count|
+--------------+------+
|        Russia|344520|
|         China|256988|
| United States|191539|
|       Germany|115246|
|United Kingdom| 93975|
|        Poland| 60132|
|         Japan| 59784|
|         Spain| 57294|
|        Turkey| 55732|
|         Italy| 51719|
+--------------+------+
only showing top 10 rows



[Stage 25:=================================================>        (6 + 1) / 7]

Registros restantes após filtro de incerteza: 6,296,146


## 5. Análises — Respostas às Perguntas

### P1 — Média Móvel de Temperatura por Década

Calculamos a temperatura média global agrupando os registros por década.
A coluna `Decada` é criada dividindo o ano por 10, truncando para inteiro e multiplicando
por 10 novamente (ex: 1987 → 1980).


In [8]:
df_decada = (
    df_filtrado
    .withColumn("Decada", (col("Year") / 10).cast("int") * 10)
    .groupBy("Decada")
    .agg(_round(avg("AverageTemperature"), 2).alias("AvgGlobalTemp"))
    .orderBy("Decada")
)

print("P1 — Temperatura média global por década:")
df_decada.show(truncate=False)

P1 — Temperatura média global por década:


[Stage 31:>                                                         (0 + 6) / 7]

+------+-------------+
|Decada|AvgGlobalTemp|
+------+-------------+
|1740  |22.47        |
|1750  |20.91        |
|1760  |20.19        |
|1770  |23.75        |
|1780  |23.21        |
|1790  |24.9         |
|1800  |25.63        |
|1810  |24.47        |
|1820  |24.65        |
|1830  |23.55        |
|1840  |22.2         |
|1850  |21.94        |
|1860  |20.52        |
|1870  |20.59        |
|1880  |19.28        |
|1890  |19.13        |
|1900  |18.8         |
|1910  |18.68        |
|1920  |18.79        |
|1930  |18.79        |
+------+-------------+
only showing top 20 rows



### P2 — Anomalias Locais: 10 Anos Mais Quentes por Continente

Para cada continente, filtramos países representativos e identificamos os 10 anos com
maior temperatura média nos últimos 50 anos (a partir de 1974).

> O dataset não possui coluna de continente, por isso usamos países representativos
> como proxy para cada região.


In [9]:
df_recente = df_filtrado.filter(col("Year") >= 1974)

continentes = {
    "America do Sul":   ["Brazil", "Argentina", "Chile", "Colombia", "Peru"],
    "America do Norte": ["United States", "Canada", "Mexico"],
    "Europa":           ["Germany", "France", "United Kingdom", "Spain", "Italy"],
    "Asia":             ["China", "India", "Japan", "Russia", "Indonesia"],
    "Africa":           ["Nigeria", "South Africa", "Ethiopia", "Egypt", "Kenya"],
}

for continente, paises in continentes.items():
    print(f"\nP2 — 10 anos mais quentes >> {continente}:")
    df_cont = (
        df_recente.filter(col("Country").isin(paises))
        .groupBy("Year")
        .agg(_round(avg("AverageTemperature"), 2).alias("AvgTemp"))
        .orderBy(col("AvgTemp").desc())
        .limit(10)
    )
    df_cont.show()


P2 — 10 anos mais quentes >> America do Sul:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2002|  21.81|
|2012|  21.79|
|1998|  21.76|
|1997|  21.69|
|2001|   21.6|
|2003|  21.59|
|2009|  21.59|
|2006|  21.57|
|2005|  21.57|
|2010|  21.55|
+----+-------+


P2 — 10 anos mais quentes >> America do Norte:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2013|  17.49|
|2012|   17.0|
|1998|  16.59|
|2006|  16.58|
|2011|  16.34|
|2005|  16.33|
|1990|  16.27|
|1999|  16.27|
|2002|  16.27|
|2007|  16.26|
+----+-------+


P2 — 10 anos mais quentes >> Europa:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2011|  12.26|
|2006|  12.07|
|2003|  12.01|
|2007|  11.97|
|1994|  11.94|
|1990|  11.92|
|2000|  11.92|
|2002|  11.91|
|1989|  11.86|
|2009|   11.8|
+----+-------+


P2 — 10 anos mais quentes >> Asia:
+----+-------+
|Year|AvgTemp|
+----+-------+
|2013|  21.53|
|2012|   20.0|
|1998|  19.96|
|2002|   19.9|
|2007|   19.9|
|2010|  19.89|
|2009|  19.86|
|2004|  19.73|
|2006|  19.72|
|1999|  19.64|
+---

### P3 — Cidades em Risco: Maior Desvio Padrão de Temperatura

O desvio padrão mede a **instabilidade climática** de uma cidade — quanto maior,
mais variável é a temperatura ao longo do tempo.

Filtramos registros a partir de 1920 (último século) e ordenamos pelo maior desvio padrão.


In [10]:
df_risk = (
    df_filtrado.filter(col("Year") >= 1920)
    .groupBy("City", "Country")
    .agg(_round(stddev("AverageTemperature"), 2).alias("Temperature_StdDev"))
    .orderBy(col("Temperature_StdDev").desc())
    .limit(10)
)

print("P3 — Top 10 cidades com maior instabilidade climática:")
df_risk.show()

P3 — Top 10 cidades com maior instabilidade climática:


[Stage 67:=========================================>                (5 + 2) / 7]

+-----------+-------+------------------+
|       City|Country|Temperature_StdDev|
+-----------+-------+------------------+
|     Yichun|  China|             14.34|
|   Tongliao|  China|             13.17|
|       Anda|  China|             12.94|
|       Fuyu|  China|             12.94|
|     Harbin|  China|             12.94|
|    Qianguo|  China|             12.94|
|      Hulan|  China|             12.94|
|Shuangcheng|  China|             12.94|
|   Honggang|  China|             12.94|
|   Longfeng|  China|             12.94|
+-----------+-------+------------------+



### P4 — Correlação de Estações em Zonas Tropicais

Calculamos a **correlação de Pearson** entre temperatura máxima e mínima mensal
em cidades localizadas entre as latitudes -23.5° e 23.5° (zona tropical).

> **Nota:** O dataset não fornece valores diários de mínima/máxima. `T_Max` e `T_Min`
> representam, respectivamente, o maior e menor valor mensal de `AverageTemperature`
> agrupados por cidade e ano.


In [11]:
df_tropicos = (
    df_filtrado
    .filter((col("Latitude") >= -23.5) & (col("Latitude") <= 23.5))
    .groupBy("City", "Year")
    .agg(
        _max("AverageTemperature").alias("T_Max"),
        _min("AverageTemperature").alias("T_Min"),
    )
)

corr_t = df_tropicos.stat.corr("T_Max", "T_Min")
print(f"P4 — Coeficiente de Correlação de Pearson (T_Max vs T_Min) nas zonas tropicais: {corr_t:.4f}")

[Stage 70:========================>                                 (3 + 4) / 7]

P4 — Coeficiente de Correlação de Pearson (T_Max vs T_Min) nas zonas tropicais: 0.4026


### P6 — Correlação entre Emissões de CO₂ e Aquecimento (Join)

Realizamos um **join** entre a base de temperatura (agregada por país/ano) e a base
de CO₂ (owid), cruzando pela chave `Country + Year`.

Em seguida calculamos a **correlação de Pearson** entre as emissões de CO₂ e a
temperatura média dos últimos 50 anos.


In [12]:
# Agregando temperatura por país e ano
df_temp_country = (
    df_filtrado.filter(col("Year") >= 1974)
    .groupBy("Country", "Year")
    .agg(avg("AverageTemperature").alias("AvgTemp"))
)

# Join das duas bases por País e Ano
df_join = df_temp_country.join(
    df_co2_clean,
    (df_temp_country["Country"] == df_co2_clean["Country_CO2"])
    & (df_temp_country["Year"] == df_co2_clean["Year_CO2"]),
    "inner",
).dropna(subset=["co2", "AvgTemp"])

print(f"Registros após join: {df_join.count():,}")
df_join.show(5)

corr_co2 = df_join.stat.corr("co2", "AvgTemp")
print(f"\nP6 — Coeficiente de Correlação de Pearson (CO₂ vs Temperatura Média): {corr_co2:.4f}")

Registros após join: 5,829


+-----------+----+------------------+-----------+--------+--------+--------------+---------+
|    Country|Year|           AvgTemp|Country_CO2|Year_CO2|     co2|co2_per_capita|total_ghg|
+-----------+----+------------------+-----------+--------+--------+--------------+---------+
|     Brazil|1997| 22.88038219696968|     Brazil|    1997| 306.949|         1.842| 2696.838|
| Kazakhstan|2007| 12.30571052631579| Kazakhstan|    2007| 225.712|        13.884|  267.572|
|     Russia|1979| 6.665930343511451|     Russia|    1979|2049.709|        14.873| 2928.791|
|     Israel|1985|19.618314814814816|     Israel|    1985|  24.613|         5.995|   30.503|
|Switzerland|2001| 8.521899999999997|Switzerland|    2001|  45.085|         6.239|   54.617|
+-----------+----+------------------+-----------+--------+--------+--------------+---------+
only showing top 5 rows



[Stage 99:=========================================>                (5 + 2) / 7]


P6 — Coeficiente de Correlação de Pearson (CO₂ vs Temperatura Média): -0.1727


### P7 — Ranking de Aceleração Térmica (Window Functions)

Calculamos a temperatura média por país e por década, depois usamos a função `lag()`
para comparar cada década com a anterior dentro do mesmo país.

A **aceleração** é a diferença entre a temperatura da década de 2010 e a de 2000.
Os 10 países com maior aceleração são listados.


In [13]:
# Temperatura média por país e década
df_decada_pais = (
    df_filtrado
    .withColumn("Decada", (col("Year") / 10).cast("int") * 10)
    .groupBy("Country", "Decada")
    .agg(avg("AverageTemperature").alias("TempDecada"))
)

# Window Function: lag para buscar a temperatura da década anterior
window_accel = Window.partitionBy("Country").orderBy("Decada")

df_accel = (
    df_decada_pais
    .withColumn("TempAnterior", lag("TempDecada", 1).over(window_accel))
    .withColumn("Aceleracao", col("TempDecada") - col("TempAnterior"))
    .filter(col("Decada") == 2010)
    .orderBy(col("Aceleracao").desc())
    .limit(10)
)

print("P7 — Top 10 países com maior aceleração térmica (2000 → 2010):")
df_accel.select("Country", _round("Aceleracao", 3).alias("Graus_Aceleracao")).show()

P7 — Top 10 países com maior aceleração térmica (2000 → 2010):


[Stage 109:================================================>        (6 + 1) / 7]

+----------+----------------+
|   Country|Graus_Aceleracao|
+----------+----------------+
|   Finland|           1.076|
|    Russia|           1.049|
|Kazakhstan|           0.778|
|    Canada|           0.767|
|      Iran|           0.761|
|      Iraq|           0.736|
|   Georgia|           0.727|
|   Armenia|           0.727|
|    Turkey|           0.663|
|Tajikistan|           0.639|
+----------+----------------+



### P8 — Previsão de Tendência com Regressão Linear (MLlib)

Utilizamos **Regressão Linear** do Spark MLlib para prever a temperatura média de
São Paulo nos 5 anos seguintes ao último registro disponível no dataset (~2013).

O modelo é treinado com os dados de **1993 a 2013** (últimos 20 anos disponíveis).
O `VectorAssembler` converte a coluna `Year` em um vetor de features, formato
exigido pelo MLlib.


In [14]:
# Dados de treino: São Paulo, 1993–2013
df_sp = (
    df_filtrado
    .filter((col("City") == "São Paulo") & (col("Year") >= 1993))
    .groupBy("Year")
    .agg(avg("AverageTemperature").alias("label"))
    .orderBy("Year")
)

print("Dados de treino (São Paulo):")
df_sp.show()

# Preparação das features para o MLlib
assembler = VectorAssembler(inputCols=["Year"], outputCol="features")
df_ml = assembler.transform(df_sp).select("features", "label")

# Treinamento do modelo de Regressão Linear
lr = LinearRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(df_ml)

print(f"Coeficiente (inclinação): {lr_model.coefficients[0]:.6f}")
print(f"Intercepto: {lr_model.intercept:.4f}")

# Previsão para 2014–2018
anos_futuros = spark.createDataFrame([(y,) for y in range(2014, 2019)], ["Year"])
anos_futuros_ml = assembler.transform(anos_futuros)
previsoes = lr_model.transform(anos_futuros_ml)

print("\nP8 — Temperatura prevista para São Paulo (2014–2018):")
previsoes.select("Year", _round("prediction", 2).alias("Temperatura_Prevista")).show()

Dados de treino (São Paulo):
+----+------------------+
|Year|             label|
+----+------------------+
|1993|20.488416666666666|
|1994|20.770999999999997|
|1995|20.608583333333332|
|1996|20.176083333333334|
|1997|          20.77125|
|1998|20.714750000000002|
|1999|20.044416666666667|
|2000|20.316666666666666|
|2001|            20.964|
|2002|21.304000000000002|
|2003|20.755666666666666|
|2004|          20.04825|
|2005|20.783166666666666|
|2006|20.603916666666667|
|2007|20.933416666666666|
|2008|20.220666666666663|
|2009|20.775583333333334|
|2010|20.716583333333332|
|2011|20.330083333333338|
|2012| 21.01891666666667|
+----+------------------+
only showing top 20 rows



26/05/26 16:35:23 WARN Instrumentation: [b571cf5e] regParam is zero, which might cause numerical instability and overfitting.
26/05/26 16:35:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/26 16:35:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Coeficiente (inclinação): -0.000099
Intercepto: 20.7900

P8 — Temperatura prevista para São Paulo (2014–2018):
+----+--------------------+
|Year|Temperatura_Prevista|
+----+--------------------+
|2014|               20.59|
|2015|               20.59|
|2016|               20.59|
|2017|               20.59|
|2018|               20.59|
+----+--------------------+



## 7. Encerramento da Sessão

Encerramos a `SparkSession` para liberar os recursos do cluster.


In [15]:
spark.stop()
print("Sessão Spark encerrada.")

Sessão Spark encerrada.
